# IsoCourt — ST-TR Training on FineBadminton-20K

Registry category: **`st_tr`** | Script: `train_st_tr.py` | Checkpoint: `badminton_model_st_tr.pth`

Reference: Plizzari et al., *Spatial Temporal Transformer Network for Skeleton-Based Action Recognition*, arXiv 2008.07404 (ICPR 2021).

**Runtime → Change runtime type → GPU (A100 recommended, T4 works)**

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU — change runtime to GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/IsoCourt/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

## 2 — Clone repo & install dependencies

In [ ]:
REPO = 'https://github.com/Navneethd8/IsoCourt.git'
BRANCH = 'main'

if not os.path.isdir('/content/IsoCourt'):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/IsoCourt
else:
    !cd /content/IsoCourt && git pull --ff-only

%cd /content/IsoCourt

In [ ]:
!pip install -q opencv-python-headless mediapipe mlflow tqdm timm transformers safetensors huggingface_hub

## 3 — Download FineBadminton-20K & prepare data

In [ ]:
DATA_DIR = '/content/IsoCourt/backend/data/FineBadminton-20K'
MERGED_JSON = '/content/IsoCourt/backend/data/transformed_combined_rounds_output_en_evals_translated.json'

if not os.path.isfile(MERGED_JSON):
    !python backend/pipelines/vlm/common/prepare_finebadminton_20k.py \
        --local-dir {DATA_DIR} --max-workers 8
else:
    print(f'Merged JSON exists: {MERGED_JSON}')

## 4 — Train ST-TR

In [ ]:
EPOCHS = 60
SAVE_PATH = os.path.join(DRIVE_CKPT, 'badminton_model_st_tr.pth')
POSE_CACHE = os.path.join(DRIVE_CKPT, 'pose_cache_staeformer.pt')

import sys
sys.path.insert(0, '/content/IsoCourt/backend')

from pipelines.training.train_st_tr import train_st_tr

train_st_tr(
    data_root='/content/IsoCourt/backend/data',
    list_file=MERGED_JSON,
    epochs=EPOCHS,
    batch_size=8,
    lr=5e-4,
    device='cuda',
    save_path=SAVE_PATH,
    pose_cache_path=POSE_CACHE,
    embed_dim=128,
    num_heads=4,
    num_layers=3,
    fusion='concat',
)

In [ ]:
print('Training complete.')
!ls -lh {DRIVE_CKPT}/*st_tr* 2>/dev/null || echo 'No checkpoints found'